# 05. Оптимизация: от градиентного спуска до AdamW

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/05_optimization.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m05). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**5.2.** Реализуйте Adam на NumPy и минимизируйте функцию Розенброка $f(x,y) = (1-x)^2 + 100(y-x^2)^2$ из точки (−1.5, 2).

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
f = lambda p: (1 - p[0])**2 + 100 * (p[1] - p[0]**2)**2
grad = lambda p: np.array([-2 * (1 - p[0]) - 400 * p[0] * (p[1] - p[0]**2), 200 * (p[1] - p[0]**2)])

p = np.array([-1.5, 2.0]); m = v = np.zeros(2)
lr, b1, b2, eps = 0.02, 0.9, 0.999, 1e-8
for t in range(1, 20001):
    g = grad(p)
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g**2
    p = p - lr * (m / (1 - b1**t)) / (np.sqrt(v / (1 - b2**t)) + eps)
print(p, f(p))   # ≈ [1, 1], f ≈ 0

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def gd_contours():
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    X, Y = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-2.5, 2.5, 200))
    Z = rosen_like(X, Y)
    for ax, lr, title in zip(axes, [0.05, 0.2, 0.24],
                             ["lr = 0.05: медленно ползёт по x", "lr = 0.2: быстро", "lr = 0.24: зигзаг поперёк долины\n(при lr > 0.25 — расходится)"]):
        ax.contour(X, Y, Z, levels=20, cmap="Blues", alpha=0.7)
        p = np.array([-4.5, 2.0]); path = [p.copy()]
        for _ in range(25):
            g = np.array([p[0], 8 * p[1]])
            p = p - lr * g
            path.append(p.copy())
            if np.abs(p).max() > 20:
                break
        path = np.array(path)
        ax.plot(path[:, 0], path[:, 1], "o-", color=C[1], ms=3, lw=1.2)
        ax.plot(0, 0, "*", color=C[2], ms=14)
        ax.set_xlim(-5, 5); ax.set_ylim(-2.5, 2.5); ax.set_title(title)
    fig.suptitle("Градиентный спуск θ ← θ − η∇L на вытянутой «долине» L = 0.5x² + 4y²", y=1.02)
    save(fig, "gradient_descent")

gd_contours()

In [ ]:
def optimizers():
    def grad(p):
        return np.array([p[0], 8 * p[1]])

    def run(kind, lr, steps=60):
        p = np.array([-4.5, 2.0]); m = np.zeros(2); v = np.zeros(2); path = [p.copy()]
        b1, b2, eps = 0.9, 0.999, 1e-8
        for t in range(1, steps + 1):
            g = grad(p)
            if kind == "SGD":
                p = p - lr * g
            elif kind == "Momentum":
                m = 0.9 * m + g; p = p - lr * m
            elif kind == "RMSProp":
                v = 0.9 * v + 0.1 * g**2; p = p - lr * g / (np.sqrt(v) + eps)
            elif kind == "Adam":
                m = b1 * m + (1 - b1) * g; v = b2 * v + (1 - b2) * g**2
                mh = m / (1 - b1**t); vh = v / (1 - b2**t)
                p = p - lr * mh / (np.sqrt(vh) + eps)
            path.append(p.copy())
        return np.array(path)

    fig, ax = plt.subplots(figsize=(9, 4.6))
    X, Y = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-2.5, 2.5, 200))
    ax.contour(X, Y, rosen_like(X, Y), levels=20, cmap="Greys", alpha=0.5)
    for (kind, lr), c in zip([("SGD", 0.05), ("Momentum", 0.006), ("RMSProp", 0.15), ("Adam", 0.3)], C):
        P = run(kind, lr)
        ax.plot(P[:, 0], P[:, 1], ".-", color=c, lw=1.5, ms=4, label=f"{kind} (lr={lr})")
    ax.plot(0, 0, "*", color="k", ms=14)
    ax.legend(loc="lower right"); ax.set_xlim(-5, 5); ax.set_ylim(-2.5, 2.5)
    ax.set_title("60 шагов: SGD ползёт по пологой оси, Momentum разгоняется (и проскакивает),\nRMSProp и Adam выравнивают масштаб шагов по осям")
    save(fig, "optimizers")

optimizers()

In [ ]:
def lr_schedule():
    steps = np.arange(10000); warm = 500; peak = 3e-4
    lr = np.where(steps < warm, peak * steps / warm,
                  1e-5 + 0.5 * (peak - 1e-5) * (1 + np.cos(np.pi * (steps - warm) / (10000 - warm))))
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.plot(steps, lr, color=C[0], lw=2)
    ax.axvspan(0, warm, color=C[3], alpha=0.15); ax.text(30, peak * 0.5, "warmup", color=C[3])
    ax.set_xlabel("шаг"); ax.set_ylabel("learning rate"); ax.set_title("Linear warmup + cosine decay — стандарт для трансформеров")
    save(fig, "lr_schedule")

lr_schedule()

In [ ]:
def l1_l2():
    c0 = np.array([1.8, 0.6])
    H = np.array([[1.5, 0.4], [0.4, 1.0]])
    loss = lambda P: np.einsum("...i,ij,...j->...", P - c0, H, P - c0)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
    X, Y = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-2, 3, 300))
    L = loss(np.dstack([X, Y]))
    t = np.linspace(0, 2 * np.pi, 4000)
    for ax, kind, col in zip(axes, ["L1", "L2"], [C[1], C[2]]):
        if kind == "L1":
            r = 1 / (np.abs(np.cos(t)) + np.abs(np.sin(t)))
        else:
            r = np.ones_like(t)
        B = np.stack([r * np.cos(t), r * np.sin(t)], 1)   # граница области ||w|| <= 1
        lb = loss(B); best = B[lb.argmin()]
        ax.fill(B[:, 0], B[:, 1], color=col, alpha=0.25)
        ax.contour(X, Y, L, levels=[lb.min() * k for k in (0.15, 0.4, 0.7)] + [lb.min(), lb.min() * 1.6],
                   colors=C[0], alpha=0.7)
        ax.plot(*best, "o", color=col, ms=10, zorder=5)
        ax.text(best[0] + 0.1, best[1] - 0.35, f"w* = ({best[0]:.2f}, {best[1]:.2f})", color=col)
        ax.set_title("L1 (Lasso): ромб — оптимум часто в углу\n→ часть весов ровно 0 (разреженность)" if kind == "L1"
                     else "L2 (Ridge): круг — оптимум на дуге\n→ веса маленькие, но не нулевые")
        ax.plot(*c0, "*", color="k", ms=12); ax.set_aspect("equal")
        ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
        ax.set_xlim(-2, 3); ax.set_ylim(-2, 3)
    save(fig, "l1_l2")

l1_l2()

In [ ]:
def convexity_saddle():
    fig = plt.figure(figsize=(15, 4.4))
    x = np.linspace(-2, 2, 300)
    ax = fig.add_subplot(1, 3, 1)
    ax.plot(x, x**2, color=C[0], lw=2.5, label="выпуклая x²")
    ax.plot(x, x**4 - 2 * x**2 + 0.3 * x, color=C[1], lw=2.5, label="невыпуклая: 2 минимума")
    ax.plot([-1.5, 1.2], [2.25, 1.44], "o--", color=C[0], lw=1)
    ax.set_ylim(-2, 4); ax.legend(fontsize=9); ax.set_title("Хорда над графиком ⇔ выпуклость")
    X, Y = np.meshgrid(np.linspace(-2, 2, 60), np.linspace(-2, 2, 60))
    for i, (Z, t) in enumerate([(X**2 + Y**2, "минимум: H ≻ 0"), (X**2 - Y**2, "седло: λ разных знаков")]):
        ax = fig.add_subplot(1, 3, i + 2, projection="3d")
        ax.plot_surface(X, Y, Z, cmap="coolwarm", alpha=0.85, linewidth=0)
        ax.scatter([0], [0], [0], color="k", s=40)
        ax.set_title(t); ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    save(fig, "convexity_saddle")

convexity_saddle()